# HostWise — Day 3: Integration
Integrates the morning tracks into one pipeline: Track A preprocessing (V2.1 vs V2.2), the Track B model (LightGBM + XGBoost, log target), Track D error analysis and the Track C prediction demo.

**Setup:** shared EUR training set, target `price_eur`, the same 5 city-stratified CV folds, MAE in EUR. The test set is used once, after selection.

**Decision rule:** V2.2 replaces V2.1 only if it improves MAE by about 0.5 EUR or more on at least 4 of 5 folds, using inputs available for new listings.

**Plan:** inputs → V2.1 vs V2.2 → error analysis → fit and save → test → prediction demo.

In [1]:
import json
import subprocess
import sys
import tempfile
from datetime import date
from pathlib import Path

import joblib
import lightgbm
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "splits").exists())
sys.path.insert(0, str(PROJECT_ROOT / "shared_foundation"))

from preprocessing import preprocessor_v21, FEATURES_V21, preprocessor_v22, FEATURES_V22

pd.set_option("display.float_format", "{:,.2f}".format)
VERSIONS = {lib.__name__: lib.__version__ for lib in [sklearn, lightgbm, xgboost, np, pd]}
print(VERSIONS)

{'sklearn': '1.6.1', 'lightgbm': '4.7.0', 'xgboost': '2.1.4', 'numpy': '1.26.4', 'pandas': '2.2.3'}


In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "hostwise_5cities.csv"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "hostwise_pipeline.joblib"
META_PATH = MODEL_DIR / "hostwise_pipeline_meta.json"
TARGET = "price_eur"
GBP_TO_EUR = 1.15
RANDOM_STATE = 42

split_config = json.loads((SPLIT_DIR / "split_config_eur.json").read_text())
df = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id": "int64"}).set_index("id")
df["price_num"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
df[TARGET] = df["price_num"] * np.where(df["city"] == "london", GBP_TO_EUR, 1.0)

train_ids = pd.read_csv(SPLIT_DIR / "train_ids_eur.csv", dtype={"id": "int64"})["id"]
train_df = df.loc[train_ids]
y = train_df[TARGET]

assert len(train_df) == split_config["n_train"], "Training rows differ from the shared foundation"
assert set(FEATURES_V21) <= set(FEATURES_V22), "V2.2 should extend V2.1"
print(f"Training rows: {len(y):,} | V2.1 inputs: {len(FEATURES_V21)} | V2.2 inputs: {len(FEATURES_V22)}")

Training rows: 123,049 | V2.1 inputs: 16 | V2.2 inputs: 17


## Inputs for new listings
A new listing has no reviews yet. This check shows which inputs depend on past stays, and whether listings without reviews are represented in training.

In [3]:
AFTER_FIRST_STAYS = ["number_of_reviews", "review_scores_rating"]  # only known once guests have stayed

inputs = pd.DataFrame({
    "in V2.1": [f in FEATURES_V21 for f in FEATURES_V22],
    "missing in train (%)": train_df[FEATURES_V22].isna().mean().mul(100).values,
    "known for a new listing": [f not in AFTER_FIRST_STAYS for f in FEATURES_V22],
}, index=FEATURES_V22)
display(inputs)

no_reviews = train_df["number_of_reviews"].eq(0)
print(f"Training listings with no reviews: {no_reviews.mean():.1%} | "
      f"rating missing among them: {train_df.loc[no_reviews, 'review_scores_rating'].isna().mean():.1%}")

,in V2.1,missing in train (%),known for a new listing
accommodates,True,0.00,True
latitude,True,0.00,True
longitude,True,0.00,True
beds,True,2.25,True
minimum_nights,True,0.00,True
hosts_time_as_host_years,True,0.00,True
bedrooms,True,17.77,True
review_scores_rating,True,15.55,False
number_of_reviews,True,0.00,False
calculated_host_listings_count,True,0.00,True


Training listings with no reviews: 15.6% | rating missing among them: 100.0%


**Result:** only the two review inputs depend on past stays, and both versions use them. Listings without reviews (15.6%) are well represented, with the rating always missing. V2.2 adds only `amenities`, which a host can provide.

## V2.1 reference
The Track B model with V2.1. It should reproduce the Track B result (90.64 EUR).

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(train_df, train_df["city"]))
FOLD_NAMES = [f"fold {i + 1}" for i in range(len(folds))]


def log_target(model):
    return TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)


FINAL_MODEL = VotingRegressor([
    ("lightgbm", log_target(LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=127,
                                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                          random_state=RANDOM_STATE, verbose=-1))),
    ("xgboost", log_target(XGBRegressor(objective="reg:squarederror", tree_method="hist",
                                        n_estimators=2000, learning_rate=0.05, max_depth=8,
                                        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE))),
])

fold_mae, oof_pred = {}, {}


def evaluate(name, preprocessor, features):
    pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("model", clone(FINAL_MODEL))])
    pred = np.clip(cross_val_predict(pipeline, train_df[features], y, cv=folds, n_jobs=1), 0, None)
    oof_pred[name] = pred
    fold_mae[name] = pd.Series([mean_absolute_error(y.iloc[idx], pred[idx]) for _, idx in folds], index=FOLD_NAMES)
    print(f"{name}: MAE {fold_mae[name].mean():.2f} EUR")


evaluate("V2.1", preprocessor_v21, FEATURES_V21)

V2.1: MAE 90.64 EUR


**Result:** V2.1 reproduces the Track B result exactly.

## V2.2
Same model and folds with V2.2 preprocessing, compared fold by fold with V2.1.

In [5]:
evaluate("V2.2", preprocessor_v22, FEATURES_V22)

paired = pd.DataFrame(fold_mae)
paired["gain (EUR)"] = paired["V2.1"] - paired["V2.2"]
display(paired)
print(f"Mean gain: {paired['gain (EUR)'].mean():.2f} EUR | folds improved: {(paired['gain (EUR)'] > 0).sum()} of 5")

V2.2: MAE 87.97 EUR


,V2.1,V2.2,gain (EUR)
fold 1,88.91,86.45,2.46
fold 2,85.97,83.55,2.41
fold 3,93.00,90.40,2.60
fold 4,90.92,88.16,2.76
fold 5,94.38,91.27,3.11


Mean gain: 2.67 EUR | folds improved: 5 of 5


**Result:** V2.2 improves MAE by 2.67 EUR, on all 5 folds, and meets the decision rule. **Selected: V2.2 preprocessing + LightGBM/XGBoost ensemble.**

## Error analysis of the selected model (Track D)
Out-of-fold predictions of the selected model (same folds as Track D, no retraining). The Track D column is the morning baseline (HGB + V2.1, 118.59 EUR), shown for reference.

In [6]:
SELECTED = "V2.2"
SELECTED_PREPROCESSOR, SELECTED_FEATURES = preprocessor_v22, FEATURES_V22

# Morning Track D results (baseline HGB + V2.1, same folds), reference only
TRACK_D_MAE = {
    "city": {"barcelona": 90.42, "london": 120.67, "madrid": 106.69, "paris": 157.85, "rome": 78.73,
             "all listings": 118.59},
    "price range": [62.07, 60.86, 64.51, 82.29, 323.63],
    "room_type": {"Entire home/apt": 127.36, "Hotel room": 113.54, "Private room": 89.47, "Shared room": 89.93},
}

abs_err = pd.DataFrame({v: (y - oof_pred[v]).abs() for v in ["V2.1", "V2.2"]})
by_group = abs_err.groupby(train_df["city"]).mean()
by_group.loc["no reviews"] = abs_err[no_reviews].mean()
by_group.loc["all listings"] = abs_err.mean()
by_group.insert(0, "Track D baseline", pd.Series(TRACK_D_MAE["city"]))
by_group

,Track D baseline,V2.1,V2.2
city,,,
barcelona,90.42,72.91,71.42
london,120.67,92.00,89.52
madrid,106.69,69.78,68.43
paris,157.85,122.21,117.89
rome,78.73,63.77,61.81
no reviews,NaN,172.69,166.81
all listings,118.59,90.64,87.97


In [7]:
oof = pd.DataFrame({"actual": y, "predicted": oof_pred[SELECTED]}).join(train_df[["room_type", "property_type"]])
oof["residual"] = oof["actual"] - oof["predicted"]          # > 0 = under-prediction
oof["abs_error"] = oof["residual"].abs()
oof["price range"] = pd.qcut(oof["actual"], 5)
oof["direction"] = np.where(oof["residual"] > 0, "under-prediction", "over-prediction")


def error_table(by, min_listings=100):
    table = oof.groupby(by, observed=True).agg(
        listings=("abs_error", "size"), MAE=("abs_error", "mean"),
        median_AE=("abs_error", "median"), mean_residual=("residual", "mean"))
    if by in TRACK_D_MAE:
        reference = TRACK_D_MAE[by]
        table.insert(1, "Track D MAE", reference if isinstance(reference, list) else pd.Series(reference))
    return table[table["listings"] >= min_listings]


display(error_table("price range"))
display(error_table("room_type"))
display(error_table("property_type").sort_values("MAE", ascending=False).head(8))
error_table("direction")

,listings,Track D MAE,MAE,median_AE,mean_residual
price range,,,,,
"(7.029, 113.85]",24651,62.07,25.44,16.49,-20.22
"(113.85, 165.47]",24596,60.86,31.84,22.60,-17.57
"(165.47, 232.875]",24584,64.51,42.06,31.02,-12.01
"(232.875, 360.0]",24651,82.29,60.63,47.35,3.67
"(360.0, 97003.05]",24567,323.63,280.27,121.48,198.28


,listings,Track D MAE,MAE,median_AE,mean_residual
room_type,,,,,
Entire home/apt,94209,127.36,97.20,40.96,31.24
Hotel room,556,113.54,70.53,38.11,21.41
Private room,27814,89.47,57.90,18.57,27.77
Shared room,470,89.93,35.95,8.74,17.43


,listings,MAE,median_AE,mean_residual
property_type,,,,
Entire villa,113,589.92,148.33,320.03
Entire townhouse,590,209.37,88.94,70.88
Entire home,5739,161.29,66.36,58.30
Room in hotel,1861,116.31,35.20,49.59
Room in boutique hotel,674,114.94,51.51,40.18
Entire serviced apartment,2050,112.11,50.58,33.49
Entire loft,1101,99.53,38.45,40.77
Entire rental unit,72890,93.00,39.89,29.47


,listings,MAE,median_AE,mean_residual
direction,,,,
over-prediction,65408,54.18,31.31,-54.18
under-prediction,57641,126.30,37.76,126.30


**Findings:** TODO — write after running the two cells above.

## Fit and save
The selected pipeline (preprocessing + model) is trained on all training rows and saved with its metadata.

In [8]:
final_pipeline = Pipeline([("preprocessor", clone(SELECTED_PREPROCESSOR)), ("model", clone(FINAL_MODEL))])
final_pipeline.fit(train_df[SELECTED_FEATURES], y)

MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(final_pipeline, MODEL_PATH, compress=3)

metadata = {
    "model": "VotingRegressor (LightGBM + XGBoost), log1p target",
    "preprocessing": SELECTED,
    "features": SELECTED_FEATURES,
    "target": "nightly price in EUR (London converted from GBP)",
    "gbp_to_eur": GBP_TO_EUR,
    "n_train": len(y),
    "trained_date": date.today().isoformat(),
    "cv_mae_eur": round(fold_mae[SELECTED].mean(), 2),
    "cv_mae_by_city_eur": by_group[SELECTED].drop(["no reviews", "all listings"]).round(2).to_dict(),
    "cv_mae_no_reviews_eur": round(by_group.loc["no reviews", SELECTED], 2),
    "versions": VERSIONS,
}
META_PATH.write_text(json.dumps(metadata, indent=2))
print(f"Saved {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB) and {META_PATH.name}")

Saved hostwise_pipeline.joblib (17.3 MB) and hostwise_pipeline_meta.json


## Reload check
The saved file is loaded in this session and in a separate, fresh Python process with only `shared_foundation` on the path. Predictions must match the in-memory pipeline.

In [9]:
sample = train_df[SELECTED_FEATURES].head(5)
expected = final_pipeline.predict(sample)

assert np.allclose(joblib.load(MODEL_PATH).predict(sample), expected), "Reload in this session differs"

with tempfile.TemporaryDirectory() as tmp:
    sample_path = Path(tmp) / "sample.joblib"
    joblib.dump(sample, sample_path)
    script = (
        "import sys, json, joblib\n"
        f"sys.path.insert(0, r'{PROJECT_ROOT / 'shared_foundation'}')\n"
        f"model = joblib.load(r'{MODEL_PATH}')\n"
        f"print(json.dumps(model.predict(joblib.load(r'{sample_path}')).tolist()))\n"
    )
    result = subprocess.run([sys.executable, "-c", script], cwd=tmp, capture_output=True, text=True)

assert result.returncode == 0, result.stderr[-1000:]
fresh = np.array(json.loads(result.stdout.strip().splitlines()[-1]))
assert np.allclose(fresh, expected), "Reload in a fresh process differs"
print("Reload check passed (same session and fresh process)")

Reload check passed (same session and fresh process)


## Test set evaluation
The saved pipeline is evaluated once on the held-out test set. The result is reported as is; the model is not changed afterwards.

In [10]:
test_ids = pd.read_csv(SPLIT_DIR / "test_ids_eur.csv", dtype={"id": "int64"})["id"]
test_df = df.loc[test_ids]
assert len(test_df) == split_config["n_test"] and not set(test_ids) & set(train_ids)

test_pred = np.clip(final_pipeline.predict(test_df[SELECTED_FEATURES]), 0, None)
test_err = (test_df[TARGET] - test_pred).abs()

test_mae = test_err.groupby(test_df["city"]).mean()
test_mae["no reviews"] = test_err[test_df["number_of_reviews"].eq(0)].mean()
test_mae["all listings"] = test_err.mean()

metadata["test_mae_eur"] = round(test_mae["all listings"], 2)
META_PATH.write_text(json.dumps(metadata, indent=2))

pd.DataFrame({"CV MAE": by_group[SELECTED], "test MAE": test_mae})

,CV MAE,test MAE
city,,
barcelona,71.42,69.10
london,89.52,147.87
madrid,68.43,57.24
paris,117.89,114.27
rome,61.81,62.86
no reviews,166.81,300.79
all listings,87.97,108.26


## Largest test errors
Diagnostic only: the official test MAE above is not changed.

In [11]:
worst = (test_df.assign(predicted=test_pred, abs_error=test_err)
         .nlargest(5, "abs_error")[["city", TARGET, "predicted", "abs_error", "number_of_reviews", "accommodates"]])
display(worst)

extreme = test_err.idxmax()
metadata.update({
    "test_mae_without_extreme_listing_eur": round(test_err.drop(extreme).mean(), 2),
    "test_median_ae_eur": round(test_err.median(), 2),
    "cv_median_ae_eur": round(abs_err[SELECTED].median(), 2),
})
META_PATH.write_text(json.dumps(metadata, indent=2))
print(f"Official test MAE: {test_err.mean():.2f} EUR | without the largest error (diagnostic only): "
      f"{metadata['test_mae_without_extreme_listing_eur']:.2f} | median AE: test "
      f"{metadata['test_median_ae_eur']:.2f}, CV {metadata['cv_median_ae_eur']:.2f}")

,city,price_eur,predicted,abs_error,number_of_reviews,accommodates
id,,,,,,
1389784758661918909,london,"606,652.60",178.30,"606,474.30",0,1
1289811437722717777,london,"68,096.10",401.63,"67,694.47",2,3
608065383770041323,paris,"57,064.00",249.38,"56,814.62",6,2
1384148619625788404,london,"15,640.00",678.94,"14,961.06",0,2
1495756143572407893,madrid,"15,000.00","1,045.07","13,954.93",0,2


Official test MAE: 108.26 EUR | without the largest error (diagnostic only): 88.54 | median AE: test 33.47, CV 33.76


**Result:** the official test MAE is **108.26 EUR**. One London listing priced 606,653 EUR/night (1 guest, no reviews, over 6× the training maximum) adds about 19.7 EUR to it; without that listing the MAE is 88.54 EUR (diagnostic only, not a replacement score). Median error is 33.47 EUR on test vs 33.76 in CV.

## Conclusions
- **Pipeline:** V2.2 preprocessing (Track A) + LightGBM/XGBoost ensemble on the log target (Track B), saved as `models/hostwise_pipeline.joblib` with metadata.
- **Selection:** CV MAE 87.97 EUR vs 90.64 with V2.1 (−2.67 EUR, all 5 folds, every city). Morning baseline: 118.59 EUR.
- **Test (used once):** MAE 108.26 EUR, inflated by one implausible price; median error matches CV (33.47 vs 33.76 EUR).
- **Error patterns (Track D):** the largest errors come from extreme prices; listings without reviews have about twice the average error.
- **Next:** agree a price ceiling on domain grounds before any re-run; implausible prices are not yet filtered from the modeling scope.

## Prediction demo (Track C)
Loads the saved pipeline from disk and predicts a nightly price. Required inputs are checked; optional ones may be left out.

In [12]:
pipeline = joblib.load(MODEL_PATH)          # the saved pipeline, not the in-memory model
meta = json.loads(META_PATH.read_text())

INPUTS = list(pipeline.feature_names_in_)
REQUIRED = [c for c in ["city", "neighbourhood_cleansed", "latitude", "longitude", "room_type",
                        "property_type", "accommodates", "amenities"] if c in INPUTS]
preprocessor = pipeline.named_steps["preprocessor"]
CAT_COLUMNS = next(cols for name, _, cols in preprocessor.transformers_ if name == "cat")
ENCODER = preprocessor.named_transformers_["cat"].named_steps["encoder"]
TEXT_COLUMNS = [c for c in INPUTS if c in CAT_COLUMNS or c in ("bathrooms_text", "amenities")]


def predict_price(listing):
    listing = {**listing, "city": str(listing.get("city", "")).strip().lower()}
    missing = [c for c in REQUIRED if listing.get(c) in (None, "")]
    if missing:
        raise ValueError(f"Missing required inputs: {missing}")
    if listing["city"] not in meta["cv_mae_by_city_eur"]:
        raise ValueError(f"Unknown city '{listing['city']}', expected one of {list(meta['cv_mae_by_city_eur'])}")
    new_listing = not listing.get("number_of_reviews")
    if new_listing:                         # no reviews yet: the rating stays empty, as in training
        listing.update(number_of_reviews=0, review_scores_rating=None)

    unseen = [f"{col} '{listing[col]}' ({'grouped as rare' if rare is not None else 'ignored'})"
              for col, seen, rare in zip(CAT_COLUMNS, ENCODER.categories_, ENCODER.infrequent_categories_)
              if listing.get(col) is not None and listing[col] not in seen]

    X = pd.DataFrame([{c: np.nan if listing.get(c) is None else listing[c] for c in INPUTS}])
    X[TEXT_COLUMNS] = X[TEXT_COLUMNS].astype(object)     # optional inputs left empty are imputed
    eur = max(float(pipeline.predict(X)[0]), 0.0)
    return {
        "price_eur": eur,
        "price_gbp": eur / meta["gbp_to_eur"] if listing["city"] == "london" else None,
        "cv_mae_eur": meta["cv_mae_no_reviews_eur"] if new_listing else meta["cv_mae_by_city_eur"][listing["city"]],
        "cv_mae_group": "listings without reviews" if new_listing else f"{listing['city'].title()} listings",
        "unseen": "; ".join(unseen),
    }


def describe(listing, result):
    size = f"{listing['accommodates']} guests" + (f", {listing['bedrooms']} bedrooms" if listing.get("bedrooms") else "")
    gbp = f" (≈ {result['price_gbp']:,.0f} GBP)" if result["price_gbp"] is not None else ""
    note = f"  \nNote: unseen {result['unseen']}." if result["unseen"] else ""
    return (f"We predict this **{listing['room_type']}** in **{str(listing['city']).title()}** ({size}) "
            f"costs about **{result['price_eur']:,.0f} EUR**{gbp} per night.  \n"
            f"*Historical CV MAE for {result['cv_mae_group']}: {result['cv_mae_eur']:,.0f} EUR "
            f"(past model performance, not a guaranteed range).*{note}")

In [13]:
paris = {
    "city": "paris",
    "neighbourhood_cleansed": "Hôtel-de-Ville",
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "accommodates": 4,
    "bedrooms": 2,
    "beds": 2,
    "bathrooms_text": "1 bath",
    "latitude": 48.8566,
    "longitude": 2.3522,
    "minimum_nights": 2,
    "review_scores_rating": 4.8,
    "number_of_reviews": 25,
    "host_is_superhost": "f",
    "calculated_host_listings_count": 1,
    "hosts_time_as_host_years": 3,
    "amenities": '["Wifi", "Kitchen", "Washer", "Elevator", "Dedicated workspace", "Heating", "Hair dryer"]',
}

checks = {
    "Paris example": paris,
    "same listing, no reviews yet": {**paris, "number_of_reviews": 0, "review_scores_rating": None},
    "unseen property type": {**paris, "property_type": "New property type"},
    "optional inputs missing": {k: v for k, v in paris.items() if k not in ("bedrooms", "beds", "host_is_superhost")},
    "London listing": {**paris, "city": "london", "neighbourhood_cleansed": "Westminster",
                       "latitude": 51.4975, "longitude": -0.1357},
}
display(pd.DataFrame([predict_price(listing) for listing in checks.values()], index=list(checks)))

try:
    predict_price({k: v for k, v in paris.items() if k != "latitude"})
except ValueError as error:
    print("Required-input check:", error)

,price_eur,price_gbp,cv_mae_eur,cv_mae_group,unseen
Paris example,374.07,NaN,117.89,Paris listings,
"same listing, no reviews yet",570.17,NaN,166.81,listings without reviews,
unseen property type,401.71,NaN,117.89,Paris listings,property_type 'New property type' (grouped as ...
optional inputs missing,258.66,NaN,117.89,Paris listings,
London listing,377.86,328.58,89.52,London listings,


Required-input check: Missing required inputs: ['latitude']


## Predict a listing
Edit the values below and run the cell. Leave `number_of_reviews` at 0 for a new listing.

In [14]:
# --- Enter the listing here ---
listing = {
    "city": "paris",
    "neighbourhood_cleansed": "Hôtel-de-Ville",
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "accommodates": 4,
    "bedrooms": 2,
    "beds": 2,
    "bathrooms_text": "1 bath",
    "latitude": 48.8566,
    "longitude": 2.3522,
    "minimum_nights": 2,
    "review_scores_rating": 4.8,
    "number_of_reviews": 25,
    "host_is_superhost": "f",
    "calculated_host_listings_count": 1,
    "hosts_time_as_host_years": 3,
    "amenities": '["Wifi", "Kitchen", "Washer", "Elevator", "Dedicated workspace", "Heating", "Hair dryer"]',
}
# ------------------------------

display(Markdown(describe(listing, predict_price(listing))))

We predict this **Entire home/apt** in **Paris** (4 guests, 2 bedrooms) costs about **374 EUR** per night.  
*Historical CV MAE for Paris listings: 118 EUR (past model performance, not a guaranteed range).*